# Motion potential 2D — investigation

Loads the checkpoint saved by `train.ipynb` and compares predicted vs true trajectories on the held-out test set. **Run `train.ipynb` first** so that `particle_2d_dataset.npz` and `best_model.pt` exist on disk.

This notebook doesn't do any training, so iterating on plots is fast.

## Colab setup

Only needed on Colab. Make sure this points at the same folder as `train.ipynb` so the `.npz` and `.pt` files are visible.

In [ ]:
# Colab-only.
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# PROJECT_DIR = '/content/drive/MyDrive/motion_potential_2d'
# if PROJECT_DIR not in sys.path:
#     sys.path.insert(0, PROJECT_DIR)
# %cd $PROJECT_DIR

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Rebuild dataset, splits, and model

The seed and fractions here must match what was used in `train.ipynb` for the test split to line up.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from models import ParticleTrajectoryDataset, TrajectoryPredictor
from train import build_splits

DATASET_PATH = "particle_2d_dataset.npz"
CHECKPOINT_PATH = "best_model.pt"

SEED = 42
TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
TRAJ_LEN = 2001

dataset = ParticleTrajectoryDataset(DATASET_PATH)
train_set, val_set, test_set = build_splits(
    dataset, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, seed=SEED,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TrajectoryPredictor(
    traj_len=TRAJ_LEN, state_dim=4, pot_latent_dim=256, hidden_dim=512,
).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

print(f"Train/val/test sizes: {len(train_set)}/{len(val_set)}/{len(test_set)}")
print(f"Device: {device}")

## 2. Helpers: un-normalize + raw potentials

`ParticleTrajectoryDataset` applies normalization for the model. For plotting we want the trajectories back in physical units, and we want the *raw* (un-normalized) potentials for the heatmap background.

In [ ]:
traj_mean = dataset.traj_mean[0]
traj_std = dataset.traj_std[0]

raw_data = np.load(DATASET_PATH)
xs = raw_data["xs"]
ys = raw_data["ys"]
raw_potentials = raw_data["potentials"]

def unnormalize_traj(traj_tensor):
    if isinstance(traj_tensor, torch.Tensor):
        traj_tensor = traj_tensor.detach().cpu().numpy()
    return traj_tensor * traj_std + traj_mean

## 3. Plot predictions on a handful of test examples

In [ ]:
NUM_EXAMPLES = 5

for i in range(NUM_EXAMPLES):
    sample = test_set[i]

    potential = sample["potential"].unsqueeze(0).to(device)
    init_state = sample["init_state"].unsqueeze(0).to(device)
    true_traj = sample["trajectory"]
    mask = sample["traj_mask"].bool()

    with torch.no_grad():
        pred_traj = model(potential, init_state)[0]

    true_traj_valid = unnormalize_traj(true_traj[mask])
    pred_traj_valid = unnormalize_traj(pred_traj[mask])

    raw_idx = test_set.indices[i]
    V = raw_potentials[raw_idx]

    plt.figure(figsize=(7, 6))
    plt.imshow(
        V.T,
        origin="lower",
        extent=[xs[0], xs[-1], ys[0], ys[-1]],
        aspect="auto",
        alpha=0.85,
    )
    plt.colorbar(label="V(x,y)")

    plt.plot(true_traj_valid[:, 0], true_traj_valid[:, 1], label="true", linewidth=2)
    plt.plot(pred_traj_valid[:, 0], pred_traj_valid[:, 1], label="predicted", linewidth=2)
    plt.scatter(true_traj_valid[0, 0], true_traj_valid[0, 1], marker="o", s=40, label="start")

    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Test example {i}: trajectory on potential heatmap")
    plt.axis("equal")
    plt.legend()
    plt.show()

## 4. Scratch space

Add further analyses here — e.g. per-sample MSE distributions, errors stratified by `is_central`, energy/angular-momentum drift of the predicted trajectories, etc.